In [ ]:
#!pip install numpy

In [ ]:
import numpy as np
from collections import Counter
import urllib.request
import zipfile
import os
import time

In [ ]:
# maths helper
def sigmoid(x):
    """Numerically stable sigmoid function to prevent overflow."""
    return np.where(x >= 0, 
                    1 / (1 + np.exp(-x)), 
                    np.exp(x) / (1 + np.exp(x)))

In [ ]:
# Data prep

class Vocabulary:
    def __init__(self, min_count=5):
        self.min_count = min_count
        self.word2id = {}
        self.id2word = {}
        self.word_counts = {}
        self.vocab_size = 0
        self.unigram_table = []

    def build_vocab(self, tokens):
        print("Building vocabulary...")
        raw_counts = Counter(tokens)
            
        current_id = 0
        for word, count in raw_counts.items():
            if count >= self.min_count:
                self.word2id[word] = current_id
                self.id2word[current_id] = word
                self.word_counts[current_id] = count
                current_id += 1
                
        self.vocab_size = len(self.word2id)
        print(f"Vocabulary built. Size: {self.vocab_size}")

    def build_unigram_table(self, table_size=1e7):
        """Builds the 3/4 power unigram table for O(1) negative sampling."""
        print("Building unigram table...")
        pow_frequency = np.array([self.word_counts[i]**0.75 for i in range(self.vocab_size)])
        probabilities = pow_frequency / pow_frequency.sum()
        table_counts = np.round(probabilities * table_size).astype(int)
        
        table = []
        for word_id, count in enumerate(table_counts):
            table.extend([word_id] * count)
        self.unigram_table = np.array(table)

    def get_negative_samples(self, num_samples):
        random_indices = np.random.randint(0, len(self.unigram_table), size=num_samples)
        return self.unigram_table[random_indices]

In [ ]:
# Neural network

class Word2VecSGNS:
    def __init__(self, vocab_size, embed_dim, learning_rate=0.025):
        self.vocab_size = vocab_size
        self.embed_dim = embed_dim
        self.lr = learning_rate
        
        # Initialization: Center weights are small random numbers. Context weights are zeros.
        self.W_center = np.random.uniform(-0.5/embed_dim, 0.5/embed_dim, (vocab_size, embed_dim))
        self.W_context = np.zeros((vocab_size, embed_dim))
        
    def train_step(self, center_id, context_id, negative_ids):
        """Executes one forward and backward pass, updating parameters."""
        
        # 1. Fetch the center vector
        v_c = self.W_center[center_id] # Shape: (d,)
        
        # 2. Setup Context Matrix (Positive + Negatives)
        context_indices = [context_id] + list(negative_ids)
        U = self.W_context[context_indices] # Shape: (K+1, d)
        
        # 3. Forward Pass
        z = np.dot(U, v_c)          # Shape: (K+1,)
        y_hat = sigmoid(z)          # Shape: (K+1,)
        
        # True labels: 1 for positive, 0 for negatives
        y_true = np.zeros(len(context_indices))
        y_true[0] = 1.0
        
        # 4. Backward Pass (Compute Gradients)
        error = y_hat - y_true      # Shape: (K+1,)
        
        # Gradient for Context Vectors: dU = outer product of error and v_c
        dU = np.outer(error, v_c)   # Shape: (K+1, d)
        
        # Gradient for Center Vector: dv_c = dot product of U.T and error
        dv_c = np.dot(U.T, error)   # Shape: (d,)
        
        # 5. Parameter Updates (SGD)
        self.W_center[center_id] -= self.lr * dv_c
        
        # loop here because negative_ids might contain duplicates. 
        # Standard slicing (self.W_context[context_indices] -= ...) ignores duplicate updates.
        for i, idx in enumerate(context_indices):
            self.W_context[idx] -= self.lr * dU[i]

In [ ]:
# training loop and data loading
def download_text8(filename="text8.zip"):
    # Downloads the text8 dataset if it doesn't exist.
    if not os.path.exists(filename) and not os.path.exists("text8"):
        print("Downloading text8 dataset...")
        urllib.request.urlretrieve("http://mattmahoney.net/dc/text8.zip", filename)
        with zipfile.ZipFile(filename, 'r') as zip_ref:
            zip_ref.extractall(".")
    
    with open("text8", "r") as f:
        return f.read().split() # Returns a flat list of words

In [ ]:

def train(tokens, window_size=5, embed_dim=100, num_negatives=5, epochs=1):
    vocab = Vocabulary(min_count=5)
    vocab.build_vocab(tokens)
    vocab.build_unigram_table()
    
    # Convert text to integers
    print("Converting text to integer IDs...")
    corpus_ids = [vocab.word2id[w] for w in tokens if w in vocab.word2id]
    
    model = Word2VecSGNS(vocab.vocab_size, embed_dim)
    
    print("Starting training...")
    total_words = len(corpus_ids)
    
    for epoch in range(epochs):
        start_time = time.time()
        
        for i, center_id in enumerate(corpus_ids):
            # Dynamic window size (optimisation)
            dynamic_window = np.random.randint(1, window_size + 1)
            
            start = max(0, i - dynamic_window)
            end = min(total_words, i + dynamic_window + 1)
            
            for j in range(start, end):
                if i != j:
                    context_id = corpus_ids[j]
                    negative_ids = vocab.get_negative_samples(num_negatives)
                    model.train_step(center_id, context_id, negative_ids)
            
            # Simple progress tracker
            if i % 100000 == 0 and i > 0:
                print(f"Epoch {epoch+1} | Processed {i}/{total_words} words...")
                
        print(f"Epoch {epoch+1} completed in {time.time() - start_time:.2f} seconds.")
        
    return model, vocab


In [ ]:
class Evaluator:
    def __init__(self, model, vocab):
        self.vocab = vocab
        # It is standard practice to use either just W_center, 
        # or the average of W_center and W_context. We will use W_center.
        self.embeddings = model.W_center
        
        # Pre-compute the magnitudes (norms) of all vectors to speed up queries
        # Adding a tiny epsilon (1e-9) prevents division by zero
        self.norms = np.linalg.norm(self.embeddings, axis=1, keepdims=True) + 1e-9
        self.normalized_embeddings = self.embeddings / self.norms

    def get_similar_words(self, word, top_k=5):
        """Returns the top_k most similar words to the given word."""
        if word not in self.vocab.word2id:
            return f"Word '{word}' not in vocabulary."
            
        word_id = self.vocab.word2id[word]
        word_vec = self.normalized_embeddings[word_id]
        
        # Vectorized Cosine Similarity: Dot product of the query vector 
        # with all normalized embeddings at once.
        similarities = np.dot(self.normalized_embeddings, word_vec)
        
        # Get the indices of the highest similarities.
        # argsort sorts ascending, so we take the last top_k+1 elements and reverse them.
        # We use top_k + 1 because the word will always be most similar to itself (score of 1.0).
        nearest_indices = np.argsort(similarities)[-(top_k + 1):][::-1]
        
        results = []
        for idx in nearest_indices:
            if idx != word_id: # Skip the query word itself
                results.append((self.vocab.id2word[idx], similarities[idx]))
                if len(results) == top_k:
                    break
                    
        return results

In [ ]:
if __name__ == "__main__":
    tokens = download_text8()
    
    # quicker testing, slice tokens 
    tokens = tokens[:100000] 
    
    # Train
    model, vocab = train(tokens, window_size=5, embed_dim=100, epochs=1)
    
    # Evaluate
    print("\n--- Evaluation ---")
    evaluator = Evaluator(model, vocab)
    
    test_words = ["one", "american", "king", "computer", "war"]
    
    for word in test_words:
        print(f"\nNearest neighbors to '{word}':")
        neighbors = evaluator.get_similar_words(word, top_k=5)
        if isinstance(neighbors, str):
            print(neighbors)
        else:
            for neighbor, score in neighbors:
                print(f"  {neighbor}: {score:.4f}")